# ETL Silver → Gold: Migração para Modelo Dimensional

Este notebook realiza a extração, transformação e carga (ETL) dos dados da camada **Silver** (tabela `lancamentos` no schema `public`) para a camada **Gold** (modelo dimensional Star Schema no schema `gold`).

## 1. Importações e Configurações

In [9]:
import psycopg2
from psycopg2 import sql
import pandas as pd
from datetime import datetime

# Configurações de conexão com o banco de dados
DB_CONFIG = {
    'host': 'localhost',
    'port': 5433,
    'database': 'filmes_db',
    'user': 'user',
    'password': 'password'
}

print("✓ Bibliotecas importadas com sucesso!")
print(f"✓ Configuração: {DB_CONFIG['database']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}")

✓ Bibliotecas importadas com sucesso!
✓ Configuração: filmes_db@localhost:5433


## 2. Criação do Schema Gold e Todas as Tabelas (usando ddl.sql)

In [10]:
# Carregar e executar o script DDL do arquivo
import os

# Caminho para o arquivo DDL
ddl_file_path = '../Data-Layer/gold/sql/ddl.sql'

try:
    # Ler o arquivo DDL
    with open(ddl_file_path, 'r', encoding='utf-8') as file:
        ddl_script = file.read()
    
    print(f"✓ Arquivo DDL carregado: {ddl_file_path}")
    print(f"✓ Tamanho do script: {len(ddl_script)} caracteres\n")
    
    # Executar o DDL
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    cursor.execute(ddl_script)
    conn.commit()
    
    print("=" * 60)
    print("✓ Schema 'gold' criado com sucesso!")
    print("✓ Tabelas dimensionais criadas:")
    print("  - gold.DIM_DISTRIBUIDORA")
    print("  - gold.DIM_FILME")
    print("  - gold.DIM_DATA_LANCAMENTO")
    print("✓ Tabela fato criada:")
    print("  - gold.FAT_LANCAMENTO")
    print("✓ Índices criados para otimização")
    print("✓ Chaves estrangeiras configuradas:")
    print("  - srk_filme_fk → DIM_FILME")
    print("  - srk_dlan_fk → DIM_DATA_LANCAMENTO")
    print("  - srk_dist_fk → DIM_DISTRIBUIDORA")
    print("=" * 60)
    
    cursor.close()
    conn.close()
    
except FileNotFoundError:
    print(f"✗ Erro: Arquivo DDL não encontrado em '{ddl_file_path}'")
    print("  Certifique-se de que o caminho está correto.")
except Exception as e:
    print(f"✗ Erro ao executar DDL: {e}")
    if conn:
        conn.rollback()
        conn.close()

✓ Arquivo DDL carregado: ../Data-Layer/gold/sql/ddl.sql
✓ Tamanho do script: 1895 caracteres

✓ Schema 'gold' criado com sucesso!
✓ Tabelas dimensionais criadas:
  - gold.DIM_DISTRIBUIDORA
  - gold.DIM_FILME
  - gold.DIM_DATA_LANCAMENTO
✓ Tabela fato criada:
  - gold.FAT_LANCAMENTO
✓ Índices criados para otimização
✓ Chaves estrangeiras configuradas:
  - srk_filme_fk → DIM_FILME
  - srk_dlan_fk → DIM_DATA_LANCAMENTO
  - srk_dist_fk → DIM_DISTRIBUIDORA


## 3. Extração de Dados da Camada Silver

In [11]:
# Extrair dados da tabela lancamentos no schema public
try:
    conn = psycopg2.connect(**DB_CONFIG)
    
    query = """
    SELECT 
        data_lancamento,
        titulo_original,
        cpb_roe,
        tipo_obra,
        pais_obra,
        publico_total,
        renda_total,
        distribuidora,
        registro_distribuidora,
        cnpj_distribuidora,
        ano_lancamento,
        mes_lancamento,
        dia_lancamento
    FROM public.lancamentos
    WHERE data_lancamento IS NOT NULL
    """
    
    df_silver = pd.read_sql_query(query, conn)
    
    conn.close()
    
    print(f"✓ Dados extraídos da camada Silver: {len(df_silver)} registros")
    print(f"✓ Colunas: {list(df_silver.columns)}")
    print(f"\n📊 Preview dos dados:")
    display(df_silver.head())
    
except Exception as e:
    print(f"✗ Erro ao extrair dados: {e}")
    if 'conn' in locals():
        conn.close()

✓ Dados extraídos da camada Silver: 6653 registros
✓ Colunas: ['data_lancamento', 'titulo_original', 'cpb_roe', 'tipo_obra', 'pais_obra', 'publico_total', 'renda_total', 'distribuidora', 'registro_distribuidora', 'cnpj_distribuidora', 'ano_lancamento', 'mes_lancamento', 'dia_lancamento']

📊 Preview dos dados:


C:\Users\alvea\AppData\Local\Temp\ipykernel_36740\384424181.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_silver = pd.read_sql_query(query, conn)


,data_lancamento,titulo_original,cpb_roe,tipo_obra,pais_obra,publico_total,renda_total,distribuidora,registro_distribuidora,cnpj_distribuidora,ano_lancamento,mes_lancamento,dia_lancamento
0,2025-07-31,aldo baldin - uma vida pela música,B2400467800000,documentário,brasil,15,396.34,bretz filmes distribuidora e produtora ltda - epp,19243.0,39.079.678/0001-47,2025,7,31
1,2025-07-31,death of a unicorn,E2500223800000,ficção,estados unidos,245,5725.40,warner bros. (south) inc.,265.0,33.015.827/0001-28,2025,7,31
2,2025-07-31,guns up,E2500153200000,ficção,estados unidos,994,18646.57,diamond films do brasil produção e distribuiçã...,22724.0,17.095.184/0001-13,2025,7,31
3,2025-07-31,materialists,E2500150300000,ficção,estados unidos,37310,851099.53,columbia tristar filmes do brasil ltda,84.0,00.979.601/0001-98,2025,7,31
4,2025-07-31,nada,B2300179900000,ficção,brasil,238,311.31,embauba filmes ltda,23638.0,15.144.532/0001-70,2025,7,31


## 4. Transformação e Carga - DIM_DISTRIBUIDORA

In [12]:
# Extrair distribuidoras únicas - usando os 3 campos como chave composta
# Já que não há chave única além da PK, usamos a combinação dos 3 campos
dim_distribuidora_completa = df_silver[['registro_distribuidora', 'distribuidora', 'cnpj_distribuidora']].drop_duplicates()

print(f"Total de distribuidoras únicas (combinação dos 3 campos): {len(dim_distribuidora_completa)}")

# Inserir na tabela DIM_DISTRIBUIDORA
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    insert_query = """
    INSERT INTO gold.DIM_DISTRIBUIDORA (registro_distribuidora, distribuidora, cnpj_distribuidora)
    VALUES (%s, %s, %s)
    RETURNING srk_dist_pk
    """
    
    registros_inseridos = 0
    for _, row in dim_distribuidora_completa.iterrows():
        cursor.execute(insert_query, (
            row['registro_distribuidora'],
            row['distribuidora'],
            row['cnpj_distribuidora']
        ))
        registros_inseridos += 1
    
    conn.commit()
    cursor.close()
    conn.close()
    
    print(f"✓ {registros_inseridos} distribuidoras inseridas em gold.DIM_DISTRIBUIDORA")
    print("✓ Chave composta: (registro + distribuidora + cnpj)")
    
except Exception as e:
    print(f"✗ Erro ao inserir distribuidoras: {e}")
    if conn:
        conn.rollback()
        conn.close()

Total de distribuidoras únicas (combinação dos 3 campos): 377
✓ 377 distribuidoras inseridas em gold.DIM_DISTRIBUIDORA
✓ Chave composta: (registro + distribuidora + cnpj)
✓ 377 distribuidoras inseridas em gold.DIM_DISTRIBUIDORA
✓ Chave composta: (registro + distribuidora + cnpj)


## 5. Transformação e Carga - DIM_FILME

In [13]:
# Extrair filmes únicos
dim_filme = df_silver[['titulo_original', 'tipo_obra', 'pais_obra', 'cpb_roe']].drop_duplicates()

print(f"Total de filmes únicos: {len(dim_filme)}")

# Verificar se há valores nulos que podem causar problemas
print(f"Valores nulos em titulo_original: {dim_filme['titulo_original'].isnull().sum()}")
print(f"Valores nulos em cpb_roe: {dim_filme['cpb_roe'].isnull().sum()}")

# Inserir na tabela DIM_FILME
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    insert_query = """
    INSERT INTO gold.DIM_FILME (titulo_original, tipo_obra, pais_obra, cpb_roe)
    VALUES (%s, %s, %s, %s)
    RETURNING srk_filme_pk
    """
    
    registros_inseridos = 0
    erros = 0
    
    for idx, row in dim_filme.iterrows():
        try:
            cursor.execute(insert_query, (
                row['titulo_original'],
                row['tipo_obra'],
                row['pais_obra'],
                row['cpb_roe']
            ))
            registros_inseridos += 1
        except Exception as e:
            erros += 1
            if erros <= 3:  # Mostrar apenas os 3 primeiros erros
                print(f"  ⚠ Erro ao inserir: {e}")
                print(f"     Dados: {row['titulo_original'][:50]}...")
    
    conn.commit()
    cursor.close()
    conn.close()
    
    print(f"✓ {registros_inseridos} filmes inseridos em gold.DIM_FILME")
    if erros > 0:
        print(f"⚠ {erros} erros encontrados durante a inserção")
    
except Exception as e:
    print(f"✗ Erro ao inserir filmes: {e}")
    import traceback
    traceback.print_exc()
    if 'conn' in locals():
        conn.rollback()
        conn.close()

Total de filmes únicos: 6495
Valores nulos em titulo_original: 0
Valores nulos em cpb_roe: 0
✓ 6495 filmes inseridos em gold.DIM_FILME
✓ 6495 filmes inseridos em gold.DIM_FILME


## 6. Transformação e Carga - DIM_DATA_LANCAMENTO

In [14]:
# Extrair datas únicas
dim_data = df_silver[['dia_lancamento', 'mes_lancamento', 'ano_lancamento']].drop_duplicates()

print(f"Total de datas únicas: {len(dim_data)}")

# Inserir na tabela DIM_DATA_LANCAMENTO
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    insert_query = """
    INSERT INTO gold.DIM_DATA_LANCAMENTO (dia, mes, ano)
    VALUES (%s, %s, %s)
    RETURNING srk_dlan_pk
    """
    
    registros_inseridos = 0
    for _, row in dim_data.iterrows():
        cursor.execute(insert_query, (
            int(row['dia_lancamento']),
            int(row['mes_lancamento']),
            int(row['ano_lancamento'])
        ))
        registros_inseridos += 1
    
    conn.commit()
    cursor.close()
    conn.close()
    
    print(f"✓ {registros_inseridos} datas inseridas em gold.DIM_DATA_LANCAMENTO")
    
except Exception as e:
    print(f"✗ Erro ao inserir datas: {e}")
    if conn:
        conn.rollback()
        conn.close()

Total de datas únicas: 1156
✓ 1156 datas inseridas em gold.DIM_DATA_LANCAMENTO
✓ 1156 datas inseridas em gold.DIM_DATA_LANCAMENTO


## 7. Carga da Tabela Fato - FAT_LANCAMENTO

In [15]:
# Popular tabela fato relacionando com as dimensões
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cursor = conn.cursor()
 
    
    # Query para inserir fatos com lookup nas dimensões
    insert_query = """
    INSERT INTO gold.FAT_LANCAMENTO (srk_filme_fk, srk_dlan_fk, srk_dist_fk, publico_total, renda_total)
    SELECT 
        f.srk_filme_pk,
        d.srk_dlan_pk,
        dist.srk_dist_pk,
        l.publico_total,
        l.renda_total
    FROM public.lancamentos l
    INNER JOIN gold.DIM_FILME f 
        ON l.titulo_original = f.titulo_original 
        AND l.cpb_roe = f.cpb_roe
    INNER JOIN gold.DIM_DATA_LANCAMENTO d 
        ON l.dia_lancamento = d.dia 
        AND l.mes_lancamento = d.mes 
        AND l.ano_lancamento = d.ano
    INNER JOIN gold.DIM_DISTRIBUIDORA dist 
        ON l.registro_distribuidora = dist.registro_distribuidora
        AND l.distribuidora = dist.distribuidora
        AND l.cnpj_distribuidora = dist.cnpj_distribuidora
    WHERE l.data_lancamento IS NOT NULL
    """
    
    cursor.execute(insert_query)
    registros_inseridos = cursor.rowcount
    conn.commit()
    
    print(f"\n✓ {registros_inseridos} registros inseridos em gold.FAT_LANCAMENTO")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"✗ Erro ao popular tabela fato: {e}")
    import traceback
    traceback.print_exc()
    if 'conn' in locals():
        conn.rollback()
        conn.close()


✓ 6653 registros inseridos em gold.FAT_LANCAMENTO
